In [2]:
import rasterio
from rasterio.mask import mask
from shapely.geometry import box
import numpy as np
import pandas as pd
import geopandas as gpd

In [ ]:
# check if rio exist in the geoboundries

# Path to the geojson file
boundary_file = "geoBoundaries-BRA-ADM1-all\geoBoundaries-BRA-ADM1.geojson"

# Load the GeoJSON file
brazil_boundaries = gpd.read_file(boundary_file)

# Inspect the available columns to find the city name column
print(brazil_boundaries.columns)

brazil_boundaries

city_name_column = "shapeName"

rio_boundary = brazil_boundaries[brazil_boundaries[city_name_column] == "Rio de Jeneiro"]

# Check if the boundary is found
if not rio_boundary.empty:
    print("Rio de Janeiro boundary found!")
else:
    raise ValueError("Rio de Janeiro boundary not found. Check the city name or column.")

rio_boundary.to_file("rio_boundary.geojson", driver="GeoJSON")

# Print the coordinates of the boundary (or process further as needed)
print(rio_boundary)

Index(['shapeName', 'shapeISO', 'shapeID', 'shapeGroup', 'shapeType',
       'geometry'],
      dtype='object')
Rio de Janeiro boundary found!
         shapeName shapeISO                  shapeID shapeGroup shapeType  \
22  Rio de Jeneiro    BR-RJ  14911670B85657526756793        BRA      ADM1   

                                             geometry  
22  POLYGON ((-44.78723 -22.42088, -44.78871 -22.4...  


In [ ]:
# Filepaths

# raster_file = "SVDNB_npp_20200101-20200131_00N060W_vcmcfg_v10_c202002111500\SVDNB_npp_20200101-20200131_00N060W_vcmcfg_v10_c202002111500.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200201-20200229_00N060W_vcmcfg_v10_c202003021200\SVDNB_npp_20200201-20200229_00N060W_vcmcfg_v10_c202003021200.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200301-20200331_00N060W_vcmcfg_v10_c202007042300\SVDNB_npp_20200301-20200331_00N060W_vcmcfg_v10_c202007042300.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200401-20200430_00N060W_vcmcfg_v10_c202006121200\SVDNB_npp_20200401-20200430_00N060W_vcmcfg_v10_c202006121200.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200501-20200531_00N060W_vcmcfg_v10_c202006221000\SVDNB_npp_20200501-20200531_00N060W_vcmcfg_v10_c202006221000.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200601-20200630_00N060W_vcmcfg_v10_c202008012300\SVDNB_npp_20200601-20200630_00N060W_vcmcfg_v10_c202008012300.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200701-20200731_00N060W_vcmcfg_v10_c202008142300\SVDNB_npp_20200701-20200731_00N060W_vcmcfg_v10_c202008142300.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200801-20200831_00N060W_vcmcfg_v10_c202009111000\SVDNB_npp_20200801-20200831_00N060W_vcmcfg_v10_c202009111000.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200901-20200930_00N060W_vcmcfg_v10_c202010082300\SVDNB_npp_20200901-20200930_00N060W_vcmcfg_v10_c202010082300.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20201001-20201031_00N060W_vcmcfg_v10_c202011050900\SVDNB_npp_20201001-20201031_00N060W_vcmcfg_v10_c202011050900.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20201101-20201130_00N060W_vcmcfg_v10_c202012092300\SVDNB_npp_20201101-20201130_00N060W_vcmcfg_v10_c202012092300.avg_rade9h.tif"
raster_file = "SVDNB_npp_20201201-20201231_00N060W_vcmcfg_v10_c202101130900\SVDNB_npp_20201201-20201231_00N060W_vcmcfg_v10_c202101130900.avg_rade9h.tif"

rio_boundary_file = "rio_boundary.geojson"

# output_csv = "viirs_20_rio_1.csv"
# output_csv = "viirs_20_rio_2.csv"
# output_csv = "viirs_20_rio_3.csv"
# output_csv = "viirs_20_rio_4.csv"
# output_csv = "viirs_20_rio_5.csv"
# output_csv = "viirs_20_rio_6.csv"
# output_csv = "viirs_20_rio_7.csv"
# output_csv = "viirs_20_rio_8.csv"
# output_csv = "viirs_20_rio_9.csv"
# output_csv = "viirs_20_rio_10.csv"
# output_csv = "viirs_20_rio_11.csv"
output_csv = "viirs_20_rio_12.csv"

# Load the Brasília boundary
rio = gpd.read_file(rio_boundary_file)
rio = rio.to_crs(epsg=4326)

# Open the VIIRS raster file
with rasterio.open(raster_file) as src:
    raster_bounds = box(*src.bounds)
    print("Raster Bounds:", src.bounds)
    print("rio Bounds:", rio.total_bounds)

    if not raster_bounds.intersects(rio.unary_union):
        raise ValueError("rio's boundary does not overlap with the raster extent.")

    # Clip the VIIRS raster file
    rio_geom_list = [feature["geometry"] for feature in rio.__geo_interface__["features"]]
    clipped_raster, clipped_transform = mask(src, rio_geom_list, crop=True)

# Extract Raster Values for Brasília
light_intensity = clipped_raster[0]
rows, cols = np.where(~np.isnan(light_intensity))
values = light_intensity[rows, cols]
x_coords, y_coords = rasterio.transform.xy(clipped_transform, rows, cols)

# Create a DataFrame
data = pd.DataFrame({
    'longitude': x_coords,
    'latitude': y_coords,
    'light_intensity': values
})

# Save the data to a CSV file
data.to_csv(output_csv, index=False)
print(f"Extracted data for rio saved to {output_csv}")


Raster Bounds: BoundingBox(left=-60.00208333335, bottom=-64.99791718665, right=59.99791762665001, top=0.00208333335)
rio Bounds: [-44.92543978 -23.44057219 -40.96355631 -20.83745118]


C:\Users\PC\AppData\Local\Temp\ipykernel_17912\1849445727.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  if not raster_bounds.intersects(rio.unary_union):


Extracted data for rio saved to D:\NU\semester_5\data_analysis\project\edited_project\viirs_rio_3_dataset.csv


In [ ]:
# output_csv = "viirs_20_rio_1.csv"
# output_csv = "viirs_20_rio_2.csv"
# output_csv = "viirs_20_rio_3.csv"
# output_csv = "viirs_20_rio_4.csv"
# output_csv = "viirs_20_rio_5.csv"
# output_csv = "viirs_20_rio_6.csv"
# output_csv = "viirs_20_rio_7.csv"
# output_csv = "viirs_20_rio_8.csv"
# output_csv = "viirs_20_rio_9.csv"
# output_csv = "viirs_20_rio_10.csv"
# output_csv = "viirs_20_rio_11.csv"
output_csv = "viirs_20_rio_12.csv"

df = pd.read_csv(output_csv)
df.head()